# Day 3 - Exploratory Data Analysis (EDA)

This notebook completes the Day 3 EDA brief for the Bluestock Mutual Fund Analytics Capstone. It covers NAV trends, AUM growth, SIP behaviour, category flows, investor demographics, geography, folios, return correlations, portfolio sectors, and additional performance context.

Report-ready PNG charts are exported to `reports/eda_charts/`.

## Day 3 Checklist

- NAV trend analysis for all 40 schemes, with 2023 rally and 2024 correction windows.
- AUM grouped bar chart by AMC and year.
- SIP inflow time series with Dec 2025 peak annotation.
- Category inflow heatmap.
- Investor age, SIP amount, and gender analysis.
- State-wise and T30/B30 geographic analysis.
- Folio count growth with milestones.
- NAV return correlation matrix for 10 selected funds.
- Portfolio sector allocation donut.
- 10 documented EDA findings with chart references.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHART_DIR = PROJECT_ROOT / "reports" / "eda_charts"

fund = pd.read_csv(RAW_DIR / "01_fund_master.csv", parse_dates=["launch_date"])
nav = pd.read_csv(PROCESSED_DIR / "clean_nav_history.csv", parse_dates=["date"])
raw_nav = pd.read_csv(RAW_DIR / "02_nav_history.csv", parse_dates=["date"])
aum = pd.read_csv(RAW_DIR / "03_aum_by_fund_house.csv", parse_dates=["date"])
sip = pd.read_csv(RAW_DIR / "04_monthly_sip_inflows.csv")
category_flow = pd.read_csv(RAW_DIR / "05_category_inflows.csv")
folio = pd.read_csv(RAW_DIR / "06_industry_folio_count.csv")
performance = pd.read_csv(PROCESSED_DIR / "clean_scheme_performance.csv")
transactions = pd.read_csv(PROCESSED_DIR / "clean_investor_transactions.csv", parse_dates=["transaction_date"])
holdings = pd.read_csv(RAW_DIR / "09_portfolio_holdings.csv", parse_dates=["portfolio_date"])
benchmark = pd.read_csv(RAW_DIR / "10_benchmark_indices.csv", parse_dates=["date"])

sip["month"] = pd.to_datetime(sip["month"])
category_flow["month"] = pd.to_datetime(category_flow["month"])
folio["month"] = pd.to_datetime(folio["month"])
transactions["month"] = transactions["transaction_date"].dt.to_period("M").dt.to_timestamp()

print("Datasets loaded for Day 3 EDA")
print({
    "funds": fund.shape,
    "clean_nav": nav.shape,
    "aum": aum.shape,
    "sip": sip.shape,
    "category_flow": category_flow.shape,
    "folios": folio.shape,
    "transactions": transactions.shape,
    "holdings": holdings.shape,
    "benchmark": benchmark.shape,
})

Datasets loaded for Day 3 EDA
{'funds': (40, 15), 'clean_nav': (64320, 3), 'aum': (90, 5), 'sip': (48, 6), 'category_flow': (144, 3), 'folios': (21, 6), 'transactions': (32778, 14), 'holdings': (322, 8), 'benchmark': (8050, 3)}


## Interactive Plotly Views

The first two cells below use Plotly, as required in the Day 3 task. Static report PNGs are shown in the chart gallery after them.

In [2]:
def short_scheme(name):
    for old in [
        " Fund - Regular Plan - Growth",
        " Fund - Direct Plan - Growth",
        " Fund - Regular - Growth",
        " Fund - Direct - Growth",
        " Fund - Growth",
        " - Regular Plan - Growth",
        " - Direct Plan - Growth",
        " - Regular - Growth",
        " - Direct - Growth",
        " - Growth",
    ]:
        name = str(name).replace(old, "")
    return name

fund_labels = fund.assign(scheme_label=fund["scheme_name"].map(short_scheme))
nav_plot = nav.merge(fund_labels[["amfi_code", "scheme_label"]], on="amfi_code", how="left")
nav_plot = nav_plot.sort_values(["amfi_code", "date"])
nav_plot["base_nav"] = nav_plot.groupby("amfi_code")["nav"].transform("first")
nav_plot["nav_index"] = nav_plot["nav"] / nav_plot["base_nav"] * 100

fig = px.line(
    nav_plot,
    x="date",
    y="nav_index",
    color="scheme_label",
    title="Interactive NAV Trend for All 40 Schemes, Indexed to 100",
    labels={"date": "Date", "nav_index": "NAV index", "scheme_label": "Scheme"},
)
fig.add_vrect(x0="2023-01-01", x1="2023-12-31", fillcolor="green", opacity=0.08, line_width=0)
fig.add_vrect(x0="2024-06-01", x1="2024-11-30", fillcolor="red", opacity=0.08, line_width=0)
fig.update_layout(height=650, legend=dict(orientation="v"))
fig.show()

In [3]:
fig = px.line(
    sip,
    x="month",
    y="sip_inflow_crore",
    markers=True,
    title="Interactive Monthly SIP Inflow Trend",
    labels={"month": "Month", "sip_inflow_crore": "SIP inflow, Rs. crore"},
)
peak = sip.loc[sip["sip_inflow_crore"].idxmax()]
fig.add_annotation(
    x=peak["month"],
    y=peak["sip_inflow_crore"],
    text=f"Peak: Rs. {peak['sip_inflow_crore']:,.0f} Cr",
    showarrow=True,
    arrowhead=2,
)
fig.update_layout(height=500)
fig.show()

## Chart Gallery

The following PNGs are exported for final report use. They are embedded here so the notebook opens as a readable EDA report.

### Chart 01: NAV Trend - All Schemes

![Chart 01](../reports/eda_charts/01_nav_trend_all_schemes.png)

### Chart 02: AUM Growth by AMC

![Chart 02](../reports/eda_charts/02_aum_growth_by_amc.png)

### Chart 03: SIP Inflow Trend

![Chart 03](../reports/eda_charts/03_sip_inflow_trend.png)

### Chart 04: Category Inflow Heatmap

![Chart 04](../reports/eda_charts/04_category_inflow_heatmap.png)

### Chart 05: Investor Age Distribution

![Chart 05](../reports/eda_charts/05_age_group_distribution.png)

### Chart 06: SIP Amount by Age Group

![Chart 06](../reports/eda_charts/06_sip_amount_by_age_group.png)

### Chart 07: Gender Split by SIP Amount

![Chart 07](../reports/eda_charts/07_gender_split_sip_amount.png)

### Chart 08: SIP Amount by State

![Chart 08](../reports/eda_charts/08_sip_amount_by_state.png)

### Chart 09: T30 vs B30 City Tier Split

![Chart 09](../reports/eda_charts/09_city_tier_sip_split.png)

### Chart 10: Folio Count Growth

![Chart 10](../reports/eda_charts/10_folio_count_growth.png)

### Chart 11: NAV Return Correlation Matrix

![Chart 11](../reports/eda_charts/11_nav_return_correlation_matrix.png)

### Chart 12: Sector Allocation Donut

![Chart 12](../reports/eda_charts/12_sector_allocation_donut.png)

### Chart 13: Return vs Risk Scatter

![Chart 13](../reports/eda_charts/13_return_vs_risk_scatter.png)

### Chart 14: Top Sharpe Funds

![Chart 14](../reports/eda_charts/14_top_sharpe_funds.png)

### Chart 15: Expense Ratio by Category and Plan

![Chart 15](../reports/eda_charts/15_expense_ratio_by_category_plan.png)

### Chart 16: Monthly Transaction Volume

![Chart 16](../reports/eda_charts/16_monthly_transaction_volume.png)

### Chart 17: Transaction Type Amount Split

![Chart 17](../reports/eda_charts/17_transaction_type_amount_split.png)

### Chart 18: Benchmark Index Trends

![Chart 18](../reports/eda_charts/18_benchmark_index_trends.png)

## 10 Key EDA Findings

1. NAV trends broadly strengthened after the 2023 rally window, with the median indexed fund line staying above its 2022 base by the end of the period (Chart 01).
2. SBI Mutual Fund is the latest AUM leader at Rs. 12.50 lakh crore, showing strong AMC-level concentration (Chart 02).
3. SIP inflows peak at Rs. 31,002 crore in Dec 2025, matching the Day 3 milestone callout (Chart 03).
4. Industry folios grew from 13.26 crore to 26.12 crore, a 97.0% increase across the covered period (Chart 10).
5. Madhya Pradesh contributes the highest SIP amount at about Rs. 2.07 crore among the states in the transaction dataset (Chart 08).
6. T30 cities contribute 66.7% of SIP amount, making city tier an important segmentation variable (Chart 09).
7. The 56+ age group has the highest average SIP ticket size at about Rs. 11,575 per SIP transaction (Chart 06).
8. Banking is the largest aggregate portfolio sector exposure at 19.2% of recorded holding weights (Chart 12).
9. ICICI Pru Liquid has the highest Sharpe ratio at 7.68, indicating the strongest risk-adjusted profile in the performance file (Chart 14).
10. The top-10 AUM fund return matrix has an average pairwise correlation of -0.00, so diversification benefits depend on category and benchmark mix rather than AMC count alone (Chart 11).

## Notes for Submission

- Cleaned NAV data is used for the continuous daily NAV trend because Day 2 forward-filled weekends and holidays.
- Raw business-day NAV is used for the return correlation matrix to avoid artificial zero-return weekends.
- Monetary chart axes use explicit units such as Rs. crore and Rs. lakh crore to avoid AUM unit confusion.
- Plotly static PNG export requires Kaleido, which is not installed in the current environment; therefore report PNGs are generated with Matplotlib/Seaborn while Plotly cells remain interactive in the notebook.